In [ ]:
!git clone -q https://github.com/shemshallah/nanlmv9.git
%cd nanlmv9
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
!mkdir -p data
!curl -sSL -o data/wt2_train.txt https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt
!python3 experiments/mamba_moe/tokenizer_bpe.py --vocab-texts data/wt2_train.txt --train-texts data/wt2_train.txt --out data/tok32k --vocab-size 32000

In [ ]:
!curl -L --retry 3 -o win_wordnet_256.bin https://github.com/shemshallah/nanlmv9/releases/download/wordnet-bytefirst/win_wordnet_256.bin
!curl -L --retry 3 -o cm_wordnet_256_ctx17.bin https://github.com/shemshallah/nanlmv9/releases/download/wordnet-bytefirst/cm_wordnet_256_ctx17.bin
!for p in aa ab ac ad ae af; do curl -L --retry 3 -o cm17.part-$p https://github.com/shemshallah/nanlmv9/releases/download/cm-prior-ctx17/cm17.part-$p; done
!cat cm17.part-* > cm_valid_full_ctx17.bin
!curl -L --retry 3 -o win_valid_full.bin https://github.com/shemshallah/nanlmv9/releases/download/cm-prior-ctx17/win_valid_full.bin
!ls -lh win_wordnet_256.bin cm_wordnet_256_ctx17.bin cm_valid_full_ctx17.bin

In [ ]:
!python3 experiments/mamba_moe/train_word_char.py \
  --data-dir data/tok32k --cm-dump cm_wordnet_256_ctx17.bin --window win_wordnet_256.bin \
  --eval-window win_valid_full.bin --eval-cm-dump cm_valid_full_ctx17.bin --eval-chunks 20 \
  --out runs/wn_d256 --steps 100000 --tokens 64 \
  --d-model 256 --char-model 64 --experts 64 --lr 2e-4 \
  --scan chunked --scan-chunk 4 --device cuda --amp --max-seconds 600 \
  --save-interval 500 --result-json runs/wn_d256/result.json

In [ ]:
!python3 experiments/mamba_moe/train_word_char.py \
  --data-dir data/tok32k --cm-dump cm_wordnet_256_ctx17.bin --window win_wordnet_256.bin \
  --eval-window win_valid_full.bin --eval-cm-dump cm_valid_full_ctx17.bin --eval-chunks 20 \
  --out runs/wn_d512 --steps 100000 --tokens 64 \
  --d-model 512 --char-model 128 --experts 64 --lr 2e-4 \
  --scan chunked --scan-chunk 4 --device cuda --amp --max-seconds 600 \
  --save-interval 500 --result-json runs/wn_d512/result.json

In [ ]:
import json
print('d256', open('runs/wn_d256/result.json').read())
print('d512', open('runs/wn_d512/result.json').read())